# M4 – Qwen2.5-7B (model public) chạy bằng Ollama trên Colab GPU

Dùng **đúng prompt + JSON schema** như trên máy (`llm.py`, `schema.py`) để so sánh công bằng với Gemini và Qwen-3B.

**Cách chạy**
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`. Khi ô số 3 hỏi file, chọn 3 file trong thư mục dự án:
   `colab/test_docs.jsonl`, `src/extractors/llm.py`, `src/schema.py`
3. Cuối cùng tải về `qwen2.5-7b_results.jsonl` → chép vào `NLP_FN/colab/`, rồi trên máy chạy:
   `python src/import_colab_results.py colab/qwen2.5-7b_results.jsonl --method qwen2.5-7b`

Nếu Colab ngắt giữa chừng: chạy lại từ đầu — ô số 4 tự bỏ qua các hoá đơn đã xong (nếu file kết quả còn trong /content).

In [ ]:
# 1) Cài và khởi động Ollama, tải model 7B (~4.7GB)
!sudo apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, httpx
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen2.5:7b
print(httpx.get("http://localhost:11434/api/version").json())

In [ ]:
# 2) Kiểm tra GPU
!nvidia-smi -L

In [ ]:
# 3) Tải lên: test_docs.jsonl, llm.py, schema.py
from google.colab import files
up = files.upload()
print(list(up))

In [ ]:
# 4) Chạy trích rút (có resume)
import json, os, sys
sys.path.insert(0, "/content")
import llm  # dùng chung prompt với máy local

OUT = "/content/qwen2.5-7b_results.jsonl"
done = set()
if os.path.exists(OUT):
    done = {json.loads(l)["id"] for l in open(OUT, encoding="utf-8")}
docs = [json.loads(l) for l in open("/content/test_docs.jsonl", encoding="utf-8")]
todo = [d for d in docs if d["id"] not in done]
print(f"Tổng {len(docs)}, đã xong {len(done)}, còn {len(todo)}")

from tqdm.auto import tqdm
with open(OUT, "a", encoding="utf-8") as f:
    for d in tqdm(todo):
        try:
            r = llm.ollama_extract(d["text"], model="qwen2.5:7b")
        except Exception as e:
            print("lỗi", d["id"], e); continue
        r["id"] = d["id"]
        f.write(json.dumps(r, ensure_ascii=False) + "
"); f.flush()

In [ ]:
# 5) Xem nhanh + tải về
rows = [json.loads(l) for l in open(OUT, encoding="utf-8")]
print(len(rows), "kết quả; JSON hợp lệ:", sum(r["valid_json"] for r in rows),
      "; ms/doc TB:", sum(r["latency_ms"] for r in rows) // len(rows))
print(rows[0]["fields"])
files.download(OUT)